# 05｜逐年分析与 2026 表现解释

本 Notebook 只读取 04 的 O2O 加算逐日结果和持有段明细，不重新计算信号。

输出包括：训练/验证/测试期逐年收益、状态分布、持有段收益与胜率、年度信号计数，以及专门解释 2026 表现的逐月拆解、状态覆盖和慢快线/规则轴诊断。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get(
    'COMPANY_SPOT_PATH',
    '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet',
).strip()
if not SPOT_TEXT or not Path(SPOT_TEXT).expanduser().is_absolute():
    raise RuntimeError('请设置 COMPANY_SPOT_PATH 为本地米筐现货的绝对路径。')
SPOT_PATH = Path(SPOT_TEXT).expanduser().resolve()
STAGE04_DIR = Path(os.environ.get(
    'ANALYSIS_04_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_04_returns'),
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'ANALYSIS_05_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_05_yearly'),
)).expanduser().resolve()
PANEL_TEXT = os.environ.get('REMOTE_ANALYSIS_PANEL_PATH', '').strip()
PANEL_PATH = Path(PANEL_TEXT).expanduser().resolve() if PANEL_TEXT else None
if not STAGE04_DIR.is_absolute() or not OUTPUT_DIR.is_absolute():
    raise RuntimeError('05 的 04 输入目录和输出目录都必须是绝对路径。')

from reproduce_remote_o2o import run_stage_05

print('本地现货：', SPOT_PATH)
print('04 输入：', STAGE04_DIR)
print('05 输出：', OUTPUT_DIR)
print('可选 1545 内部面板：', PANEL_PATH if PANEL_PATH else '未显式指定，将从冻结包现货重建')

## 1. 读取 04 并生成逐年分析

In [ ]:
metadata = run_stage_05(SPOT_PATH, STAGE04_DIR, OUTPUT_DIR, PANEL_PATH)
annual_adj = pd.read_csv(OUTPUT_DIR / '逐年_加入四个反转分析.csv', encoding='utf-8-sig')
annual_raw = pd.read_csv(OUTPUT_DIR / '逐年_原始三状态分析.csv', encoding='utf-8-sig')
display(annual_raw)
display(annual_adj)
print('2026 摘要：', metadata['year_2026'])

## 2. 为什么 2026 表现好

这里不把“表现好”归因于单一指标，而是同时查看：O2O 日收益、相对指数超额、方向覆盖天数、每月贡献和冻结引擎的慢快线/规则轴/四维状态。

In [ ]:
monthly_2026 = pd.read_csv(OUTPUT_DIR / '2026_逐月表现分解.csv', encoding='utf-8-sig')
detail_2026 = pd.read_csv(OUTPUT_DIR / '2026_逐日表现分解.csv', encoding='utf-8-sig', parse_dates=['实际执行日'])
display(monthly_2026)
display(detail_2026.tail(15))
print('2026 调整后加算收益：', monthly_2026['adjusted_return_pct'].sum(), '%')
print('2026 调整后相对指数超额：', monthly_2026['adjusted_excess_pct'].sum(), '%')
print('2026 调整后方向持有日：', int(detail_2026['调整后三状态'].ne(0).sum()))

## 3. 检查 04→05 的口径没有被改变

In [ ]:
stage04_detail = pd.read_csv(STAGE04_DIR / 'O2O加算逐日收益与状态.csv', encoding='utf-8-sig', parse_dates=['实际执行日', '推定形成日'])
if not stage04_detail['形成日早于执行日'].all():
    raise AssertionError('04 中存在形成日不早于执行日的行')
if (stage04_detail['O2O可评价'] & stage04_detail['执行日O2O'].isna()).any():
    raise AssertionError('04 将无 O2O 的行错误标记为可评价')
latest = stage04_detail.iloc[-1]
print('最新信号形成日：', latest['推定形成日'])
print('最新信号执行日：', latest['实际执行日'])
print('最新信号是否已有下一开盘价：', bool(latest['O2O可评价']))
print('05 只读取 04 结果，口径检查通过。')

输出图中，`15`—`19` 是冻结 1545 内部状态、慢快线背离、年度表现汇总、状态对比和全周期市场环境诊断；若要与云桌面截图完全同截止日，应给 `COMPANY_SPOT_PATH` 指向同一份已更新到最新交易日的本地现货。